# Binary Fault Classification Baseline

This notebook trains the binary baseline without this project's physics-consistency loss term. It uses the same Paderborn windows, original engineered features, grouped split, class weighting, physics weight, and learning rate as `Binary_Classification_PhysicsWeight.ipynb`. The controlled difference is `use_physics_loss=False`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
!mkdir -p /kaggle/working/repo
!cp -r "/kaggle/input/datasets/anaranyosarkar27/pinn-motor-fault-code/induction-motor-fault-classification-via-PINN" /kaggle/working/repo/project
!pip install -e /kaggle/working/repo/project --break-system-packages -q

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo/project/src")
import pinn_motor_fault
print("Package imported successfully")

In [ ]:
from pinn_motor_fault.fast_train import compute_class_weights, fast_fit
from pinn_motor_fault.features import PhysicsFeatureExtractor
from pinn_motor_fault.model import PhysicsInformedNN
from pinn_motor_fault.paderborn import load_paderborn_windows
from pinn_motor_fault.results import write_evaluation_artifacts
from pinn_motor_fault.train import stratified_group_split

DATA_DIR = Path("/kaggle/input/datasets/dippatel03/paderborn-db")
MODEL_PATH = Path("/kaggle/working/models/paderborn_binary_baseline.npz")
RESULTS_DIR = Path("/kaggle/working/reports/results_binary_baseline")
BINARY_CLASS_NAMES = ("healthy", "faulty")

windows, original_labels, sources = load_paderborn_windows(
    data_dir=DATA_DIR,
    window_size=4096,
    stride=2048,
    signal_preference="vibration",
    max_windows_per_file=30,
)
labels = np.where(original_labels == "healthy", "healthy", "faulty")
print("Original labels:", dict(zip(*np.unique(original_labels, return_counts=True))))
print("Binary labels:", dict(zip(*np.unique(labels, return_counts=True))))

batch = PhysicsFeatureExtractor().transform(windows, sources)
binary_physics_targets = np.column_stack([
    batch.physics_targets[:, 0],
    batch.physics_targets[:, 1:].sum(axis=1),
])

train_idx, test_idx = stratified_group_split(labels, sources, test_fraction=0.25, seed=17)
class_weights = compute_class_weights(labels[train_idx], BINARY_CLASS_NAMES)
print("Train/test windows:", train_idx.size, test_idx.size)
print("Class weights:", class_weights)

In [ ]:
# Baseline: ordinary binary cross-entropy training.
# The hyperparameters remain identical to the physics-weight notebook, but the physics-consistency loss term is disabled.
feature_names = batch.feature_names
model = PhysicsInformedNN(
    input_dim=batch.features.shape[1],
    hidden_dim=48,
    physics_weight=0.1,
    learning_rate=0.1,
    use_physics_loss=False,
    class_names=BINARY_CLASS_NAMES,
)
history = fast_fit(
    model,
    batch.features[train_idx],
    labels[train_idx],
    binary_physics_targets[train_idx],
    batch.features[test_idx],
    labels[test_idx],
    binary_physics_targets[test_idx],
    epochs=60,
    batch_size=64,
    class_weights=class_weights,
)
model.save(MODEL_PATH)
print(f"Saved baseline model: {MODEL_PATH}")

In [ ]:
test_predictions = model.predict(batch.features[test_idx])
test_probabilities = model.predict_proba(batch.features[test_idx])
train_loss, train_accuracy = model.loss_and_accuracy(
    batch.features[train_idx], labels[train_idx], binary_physics_targets[train_idx]
)
test_loss, test_accuracy = model.loss_and_accuracy(
    batch.features[test_idx], labels[test_idx], binary_physics_targets[test_idx]
)
settings = {
    "model_path": str(MODEL_PATH),
    "data_dir": str(DATA_DIR),
    "sample_count": int(test_idx.size),
    "train_sample_count": int(train_idx.size),
    "test_sample_count": int(test_idx.size),
    "unique_train_files": int(len(set(sources[index] for index in train_idx))),
    "unique_test_files": int(len(set(sources[index] for index in test_idx))),
    "window_size": 4096,
    "stride": 2048,
    "signal": "vibration",
    "split_strategy": "stratified_group_by_mat_file",
    "feature_set": "PhysicsFeatureExtractor original engineered features",
    "algorithm": "standard binary classifier",
    "use_physics_loss": False,
    "epochs": 60,
    "max_windows_per_file": 30,
    "physics_weight": 0.1,
    "learning_rate": 0.1,
    "train_loss": float(train_loss),
    "test_loss": float(test_loss),
    "class_weights": class_weights,
}
metrics = write_evaluation_artifacts(
    output_dir=RESULTS_DIR,
    labels=labels[test_idx],
    predictions=test_predictions,
    probabilities=test_probabilities,
    sources=[sources[index] for index in test_idx],
    model=model,
    feature_names=feature_names,
    windows=windows[test_idx],
    signal_name="vibration",
    settings=settings,
    training_history=history,
)
test_X = batch.features[test_idx]
test_y = labels[test_idx]
results_dir = str(RESULTS_DIR)
print(f"Baseline test accuracy: {test_accuracy:.4f}")
print(f"Saved results: {RESULTS_DIR}")

In [ ]:
import json
from IPython.display import SVG, display

print(pd.read_csv(f"{results_dir}/confusion_matrix.csv", index_col=0))
metrics = json.load(open(f"{results_dir}/metrics.json"))
print(json.dumps(metrics["class_metrics"], indent=2))
display(SVG(f"{results_dir}/figures/confusion_matrix.svg"))
display(SVG(f"{results_dir}/figures/training_curves.svg"))

In [ ]:
# SHAP feature importance for the predicted faulty probability.
%pip install shap -q
import shap

background_size = min(50, len(train_idx))
explain_size = min(100, len(test_X))
background = batch.features[train_idx[:background_size]]
explain_X = test_X[:explain_size]

def predict_faulty_probability(values):
    values = np.asarray(values, dtype=np.float64)
    return model.predict_proba(values)[:, 1]

explainer = shap.Explainer(
    predict_faulty_probability,
    background,
    feature_names=feature_names,
)
shap_explanation = explainer(explain_X)
shap_values = np.asarray(shap_explanation.values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]

shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.mean(np.abs(shap_values), axis=0),
}).sort_values("mean_abs_shap", ascending=False)
print(shap_importance.head(20))
shap_importance.to_csv(f"{results_dir}/shap_feature_importance.csv", index=False)

shap.summary_plot(
    shap_values,
    explain_X,
    feature_names=feature_names,
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(f"{results_dir}/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()